# Lab 04 · Apache XTable: Hudi → Iceberg

> One COW table, an explicit metadata sync, and two native readers checking the same rows.

This OSS Spark experiment synchronizes a Hudi source to an Iceberg read view with Apache XTable 0.4.0-incubating. It compares a canonical digest and records independent Hudi/Iceberg coordinates.

**Version gate:** XTable 0.4.0's released runtime is compiled against Hudi 0.14.0 and supports Spark 3.4/3.5. The Hudi 0.14 bundle is for Spark 3.4, so this lab has its own small environment file. The main Delta/Iceberg/Hudi labs continue using the separate shared Spark 3.5 stack. This avoids silently mixing incompatible format APIs. No Docker or remote catalog is required.


## 1 · Start the matched XTable runtime


In [ ]:
from pyspark.sql import SparkSession,functions as F
from pathlib import Path
from datetime import datetime
import hashlib,shutil,tempfile
ROOT=Path(tempfile.mkdtemp(prefix="morlocks-xtable-")).resolve(); PATH=(ROOT/"events").as_uri()
packages=",".join(["org.apache.hudi:hudi-spark3.4-bundle_2.12:0.14.0","org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.9.2","org.apache.xtable:xtable-spark-runtime_2.12:0.4.0-incubating"])
extensions=",".join(["org.apache.spark.sql.hudi.HoodieSparkSessionExtension","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"])
b=(SparkSession.builder.master("local[2]").appName("Morlocks-XTable-Hudi-Iceberg").config("spark.ui.enabled","false").config("spark.sql.shuffle.partitions","2").config("spark.serializer","org.apache.spark.serializer.KryoSerializer").config("spark.kryo.registrator","org.apache.spark.HoodieSparkKryoRegistrar").config("spark.sql.extensions",extensions).config("spark.sql.catalog.xtable_iceberg","org.apache.iceberg.spark.SparkCatalog").config("spark.sql.catalog.xtable_iceberg.type","hadoop").config("spark.sql.catalog.xtable_iceberg.warehouse",(ROOT/"warehouse").as_uri()).config("spark.jars.packages",packages))
spark=b.getOrCreate(); spark.sparkContext.setLogLevel("ERROR")
def frame(rows): return spark.createDataFrame(rows,"id LONG, name STRING, segment STRING, balance DOUBLE, event_time TIMESTAMP")
opts={"hoodie.table.name":"morlock_xtable","hoodie.datasource.write.table.type":"COPY_ON_WRITE","hoodie.datasource.write.operation":"upsert","hoodie.datasource.write.recordkey.field":"id","hoodie.datasource.write.precombine.field":"balance"}
frame([(1,"Ada","bronze",100.,datetime(2026,10,24,8)),(2,"Babbage","silver",200.,datetime(2026,10,24,8,5))]).write.format("hudi").options(**opts).mode("overwrite").save(PATH)
def latest_instant(): return str(spark.read.format("hudi").load(PATH).select(F.max("_hoodie_commit_time")).first()[0])
def sync():
 targets=spark._jvm.java.util.ArrayList(); targets.add("ICEBERG")
 spec=spark._jvm.org.apache.xtable.spark.TableSyncSpec.builder().key("morlock_xtable").basePath(PATH).sourceFormat("HUDI").targets(targets).build()
 return spark._jvm.org.apache.xtable.spark.XTableSyncService().sync(spec,spark._jsc.hadoopConfiguration())
h0=latest_instant(); print("Hudi source instant",h0,"sync result",sync())


## 2 · Commit a late event and synchronize the new view

The event is dated earlier than the new commit. XTable sync is still a separate operation after the Hudi write.


In [ ]:
frame([(2,"Babbage Lovelace","gold",250.,datetime(2026,10,24,8,5)),(99,"Larry Latecomer","time-traveller",42.42,datetime(2026,10,20,9,15))]).write.format("hudi").options(**opts).mode("append").save(PATH)
h1=latest_instant(); print("Hudi source instant",h1,"sync result",sync()); assert h0<h1
hudi=spark.read.format("hudi").load(PATH)
iceberg=spark.read.format("iceberg").load(PATH)
def digest(d):
 rows=d.select("id","name","segment","balance",F.date_format("event_time","yyyy-MM-dd HH:mm:ss").alias("event_time")).orderBy("id").collect()
 return len(rows),hashlib.sha256("\n".join("|".join(str(v) for v in r) for r in rows).encode()).hexdigest()
source,target=digest(hudi),digest(iceberg)
print("Hudi source:",source)
print("Iceberg target view:",target)
assert source==target and source[0]==3
assert iceberg.where("id=99").select("event_time").first()[0].date().isoformat()=="2026-10-20"
print("PASS: Hudi and Iceberg readers agree after sync; the late event remains late in event-time.")


## 3 · Compare native histories

A source instant and a target snapshot ID identify different table-native history machines. Keep the actual mapping alongside when sync completed.


In [ ]:
print("Hudi instants:",h0,h1)
spark.read.format("iceberg").load(PATH+"#snapshots").select("snapshot_id","parent_id","committed_at","operation").show(truncate=False)
print("Evidence mapping: Hudi source instant -> XTable sync completion -> Iceberg snapshot ID.")


## Conclusions · XTable

**Strengths:** XTable can publish format-native metadata for readers using a different table format without copying the Parquet data files. This run demonstrates Hudi COW → Iceberg with independent native readers.

**Trade-offs:** run and monitor a sync after the source commit; a target can lag or fail while the source is already newer. It is not a multi-format transaction.

**Version limitation observed:** XTable 0.4.0 fails with Hudi 0.15+'s removed API. This notebook pins the runtime's older Hudi/Spark 3.4 combination; use the documented compatibility matrix for other pairs. Delta target support on Spark 3.5 uses Delta Kernel and should be verified against the exact 0.4 runtime; this lab deliberately isolates the proven Hudi → Iceberg route.

**Scope:** the test covers COW and simple scalar fields only. XTable does not capture Hudi MOR log files or Delta/Iceberg delete vectors in these views. Test schema evolution, partitioning, deletes, retention, and target time travel before using them. Keep one authoritative writer and record source/target coordinates independently.


In [ ]:
spark.stop(); shutil.rmtree(ROOT,ignore_errors=True); print("PASS: temporary XTable lab cleaned up")
